# RAG: 외부 문서를 검색해 근거 기반 답변 만들기

RAG(Retrieval-Augmented Generation, 검색 증강 생성)는 질문과 관련된 외부 문서를 먼저 찾고, 그 문서를 LLM의 답변 근거로 함께 전달하는 방식이다. 모델을 다시 학습하지 않고도 사내 문서나 최신 자료를 사용할 수 있다.

### 사용하는 이유

- 모델이 학습하지 않은 문서도 답변 근거로 사용할 수 있다.
- 검색된 문서와 답변을 직접 대조할 수 있다.
- 문서의 출처와 페이지를 인용으로 제시할 수 있다.

### 반드시 확인할 점

- RAG가 환각을 완전히 없애지는 못한다.
- 잘못 나눈 chunk나 관련 없는 검색 결과는 부정확한 답변으로 이어질 수 있다.
- 최종 답변과 함께 검색된 `Document`의 본문, metadata, 인용(citation)을 확인해야 한다.


## RAG의 두 단계

이 실습은 **2-step RAG**를 사용한다. 문서를 미리 준비하는 인덱싱과 질문이 들어온 뒤 실행되는 검색·생성을 순서대로 분리한다.

### 1단계: 인덱싱

`외부 PDF → Document → chunk → embedding vector → Vector Store`

인덱싱은 외부 문서를 의미 기반으로 검색할 수 있게 변환하고 저장하는 과정이다.

- `Document`: 본문 `page_content`와 출처 `metadata`를 함께 담는 객체이다.
- `chunk`: 긴 Document를 나눈 실제 검색 단위이다.
- `Embedding Model`: 텍스트를 숫자 배열로 변환하는 모델이다.
- `embedding vector`: Embedding Model이 만든 숫자 배열 결과이다.
- `Vector Store`: vector와 원문 Document를 연결해 저장하고 검색한다.

아래 도식은 PDF가 Chroma Vector Store에 등록되는 순서이다.

```mermaid
flowchart LR
    A["외부 PDF"] --> B["PDF 읽기"]
    B --> C["list[Document]"]
    C --> D["Text Splitter"]
    D --> E["list[chunk Document]"]
    E --> F["Embedding Model"]
    F --> G["embedding vectors"]
    E --> H["Vector Store"]
    G --> H
```

### 2단계: 검색과 생성

`질문 → Retriever → list[Document] → context → Prompt → LLM 답변`

질문이 들어오면 관련 문서를 검색하고 그 문서를 근거로 답변을 생성한다.

- `Retriever`: 질문을 받아 관련 `list[Document]`를 반환한다. 답변은 만들지 않는다.
- `context`: 검색된 본문과 출처를 프롬프트에 넣기 위해 합친 문자열이다.
- `LLM 답변`: 질문과 context를 함께 읽고 만든 문자열이다.
- `인용(citation)`: 답변의 근거가 된 Document의 출처 표기이다.

아래 도식은 질문이 답변으로 바뀌는 순서이다.

```mermaid
flowchart LR
    Q["사용자 질문"] --> R["Retriever 호출"]
    R --> E["질문 Embedding"]
    E --> V["Vector Store 유사도 검색"]
    V --> D["list[Document]"]
    D --> C["context 문자열"]
    Q --> P["Prompt"]
    C --> P
    P --> L["Chat Model / LLM"]
    L --> A["근거 기반 답변"]
```

구성 요소와 RAG 구조는 [LangChain Retrieval 공식 문서](https://docs.langchain.com/oss/python/deepagents/retrieval)에서 확인할 수 있다.


## 실습 환경에 필요한 패키지 설치

RAG의 구성 요소는 기능별 패키지로 나뉘어 있다. 현재 Jupyter 커널에 다음 패키지를 설치한다.

- `langchain`: LangChain의 상위 API를 제공한다.
- `langchain-chroma`: Chroma Vector Store 연동을 제공한다.
- `langchain-openai`: OpenAI Embedding과 Chat Model 연동을 제공한다.
- `langchain-text-splitters`: Document 분할기를 제공한다.
- `pypdf`: PDF 페이지와 텍스트를 읽는다.
- `gdown`: Google Drive의 공개 파일을 내려받는다.
- `python-dotenv`: `.env`의 설정을 환경 변수로 불러온다.

`%pip`는 현재 Jupyter 커널의 Python 환경에 설치한다. 설치 후 import 오류가 계속되면 커널을 재시작한다. 설치 과정에는 네트워크가 필요하지만 OpenAI API 비용은 발생하지 않는다.


In [1]:
%pip install -U langchain langchain-chroma langchain-openai langchain-text-splitters pypdf gdown python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### `.env`에서 인증 설정 불러오기

`.env`는 API 키와 모델 설정을 코드 밖에 보관하는 파일이다. `find_dotenv()`로 파일을 찾고 `load_dotenv()`로 현재 커널의 환경 변수에 등록한다.

- `OPENAI_API_KEY`: 임베딩과 Chat Model 호출에 필요한 필수 설정이다.
- `OPENAI_CHAT_MODEL`: 사용할 Chat Model을 바꾸는 선택 설정이다.
- `LANGSMITH_*`: 추적이 필요할 때만 사용하는 선택 설정이다.

이 셀은 키 값을 출력하지 않는다. LangSmith 설정이 없어도 RAG의 기본 실행 흐름은 계속 진행된다.


In [2]:
import os
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError('.env 파일이 없음')

load_dotenv(dotenv_path, override=False)

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY 없음')

CHAT_MODEL_NAME = os.getenv('OPENAI_CHAT_MODEL', 'gpt-5.6-luna')
print('준비완료')


준비완료


## 1. 인덱싱: PDF를 검색 가능한 Vector Store로 바꾸기

인덱싱은 질문을 받기 전에 외부 문서를 검색 가능한 구조로 준비하는 과정이다.

### 처리 순서

`외부 PDF → list[Document] → list[chunk Document] → embedding vectors → Chroma Vector Store`

- PDF를 페이지별 `Document`로 읽는다.
- 각 Document를 작은 chunk로 나눈다.
- chunk 본문을 embedding vector로 변환한다.
- vector와 원문·metadata를 Chroma에 함께 저장한다.


### 백설공주 PDF 내려받기

`gdown`으로 공개 Google Drive 파일을 내려받는다.

- 입력: 공개 파일 ID이다.
- 출력: 현재 작업 디렉터리의 `snow-white.pdf`이다.
- 다음 사용처: `PdfReader`가 이 파일을 페이지별로 읽는다.

이 셀은 네트워크를 사용한다. 실행 후 파일명과 저장 경로를 확인한다.


In [3]:
!gdown 1vRwNfV1WwoFFKqj9KoQdQGSdbTGkwwH- -O snow-white.pdf


Downloading...
From: https://drive.google.com/uc?id=1vRwNfV1WwoFFKqj9KoQdQGSdbTGkwwH-
To: C:\SKN_AI\09_llm\06_rag\snow-white.pdf

  0%|          | 0.00/148k [00:00<?, ?B/s]
100%|██████████| 148k/148k [00:00<00:00, 2.00MB/s]


### PDF를 페이지별 Document로 읽기

`PdfReader`는 PDF 페이지를 읽고 `extract_text()`로 본문 문자열을 꺼낸다. 각 페이지는 LangChain `Document` 하나로 변환한다.

### Document에 저장할 값

- `page_content`: 페이지에서 추출한 본문 문자열이다.
- `metadata['source']`: 원본 PDF 경로이다.
- `metadata['page']`: 0부터 시작하는 내부 페이지 번호이다.
- `metadata['page_label']`: 사람이 보는 1부터 시작하는 페이지 번호이다.
- `metadata['total_pages']`: PDF 전체 페이지 수이다.

결과인 `docs`는 페이지별 `Document` 목록이다. 다음 Text Splitter가 본문을 나누면서 metadata도 함께 복사한다.



In [6]:
from pathlib import Path

from langchain_core.documents import Document
from pypdf import PdfReader

pdf_path = Path('snow-white.pdf')
if not pdf_path.exists():
    raise FileNotFoundError('snow-white.pdf가 없다. 앞의 다운로드 셀을 먼저 실행한다.')

#  PDF 읽어오기
reader = PdfReader(pdf_path)
total_pages = len(reader.pages) # 전체 페이지 수

# PDF를 한 페이지씩 읽어와 LangChain Document 객체로 변환
# -> List[Document]에 추가
docs: list[Document] = []

for page_index, page in enumerate(reader.pages):

    # 현재 페이지의 텍스트를 추출, 없으면 '' (빈칸)
    page_text = page.extract_text() or ''

    docs.append(
        Document(
            page_content=page_text, # 페이지 내용
            metadata={
                'source': str(pdf_path),
                'page': page_index,
                'page_label': str(page_index + 1),
                'total_pages': total_pages,
            }
        )
    )

print('Document 수:', len(docs))
print(docs[0])


Document 수: 6
page_content='백설공주
옛날 어느 왕국에 공주님이 태어났어요.
“어쩜 이렇게 어여쁠까? 살결이 눈처럼 하얗구나. 백
설공주라고 불러야겠다.”
왕과 왕비는 갓 태어난 딸을 보며 기뻐했어요.
하지만 기쁨도 잠시, 왕비는 곧 세상을 떠나고 말았어
요.
' metadata={'source': 'snow-white.pdf', 'page': 0, 'page_label': '1', 'total_pages': 6}


## Document Transformer와 chunk

`Document Transformer`는 Document를 검색 목적에 맞게 바꾸는 구성 요소이다. RAG에서는 긴 문서를 작은 **chunk**로 나누는 Text Splitter가 대표적인 Transformer이다.

### chunk가 필요한 이유

- 문서 전체를 하나로 저장하면 여러 주제가 한 벡터에 섞일 수 있다.
- chunk가 너무 작으면 문장 사이의 맥락이 끊길 수 있다.
- 적절한 크기로 나누면 질문과 관련된 부분을 더 정확히 찾을 수 있다.

### 분할 설정

- `chunk_size`: chunk 하나의 목표 최대 길이이다.
- `chunk_overlap`: 경계의 문맥을 이어 주기 위해 인접 chunk에 반복할 목표 길이이다.
- `separators`: 어떤 경계를 먼저 지키며 나눌지 정하는 우선순위이다.

`RecursiveCharacterTextSplitter`는 큰 의미 경계를 먼저 시도하고, 목표 크기에 맞지 않으면 더 작은 경계로 나눈다.

- `split_text()` 반환값: `list[str]`이다.
- `split_documents()` 반환값: metadata가 유지된 `list[Document]`이다.


### 한글 문자열에서 chunk 크기와 겹침 확인하기

먼저 짧은 문자열로 분할 규칙을 확인한다.

- 입력: 한글 문자열 하나이다.
- 설정: 최대 30자, 겹침 10자이다.
- 출력: 문자열 chunk 목록 `list[str]`이다.
- 확인: 인접 chunk에 경계 문구가 반복되는지 살펴본다.



In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = (
    "대한민국의 역사는 매우 길고 다양하다. "
    "고조선 부터 시작해서 삼국시대, 고려, 조선, 현대에 이르기까지 "
    "수많은 사건과 인물이 존재한다."
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=30,
    chunk_overlap=10
)

chunks = splitter.split_text(text)
print(len(chunks))
print(chunks)


4
['대한민국의 역사는 매우 길고 다양하다. 고조선 부터', '고조선 부터 시작해서 삼국시대, 고려, 조선, 현대에', '조선, 현대에 이르기까지 수많은 사건과 인물이', '사건과 인물이 존재한다.']


### 구분자 우선순위를 지정해 영문 문서 나누기

`separators`는 분할할 경계의 우선순위를 정한다.

1. 빈 줄 `\n\n`로 문단 경계를 먼저 찾는다.
2. 줄바꿈 `\n`을 찾는다.
3. 공백으로 단어 경계를 찾는다.
4. 필요한 경우 개별 문자 단위로 나눈다.

최대 50자와 10자 겹침을 사용한다. 결과에서 문단과 단어 경계가 가능한 한 유지되는지 확인한다.


In [14]:
text = """
This is a long document. It contains multiple paragraphs, sentences, and words.
The purpose of this text is to demonstrate how to split text into smaller chunks.

This is useful for indexing, searching, and processing large documents.
Each chunk will have a defined maximum size and may overlap with adjacent chunks.
"""

splitter = RecursiveCharacterTextSplitter(
    # 청크당 최대 50자, 가능한 앞쪽 구분자 경계를 우선 보존한다
    chunk_size=50,

    # chunk 앞 약 10글자를 반복해서 경계에서 끊긴 문맥을 보완
    chunk_overlap=10,

    # 문단 -> 문장 -> 단어 -> 문자 순서로 분할 가능한 경계를 찾게함
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_text(text)
print(len(chunks))

for chunk in chunks:
    print(len(chunk), chunk)

8
45 This is a long document. It contains multiple
42 multiple paragraphs, sentences, and words.
49 The purpose of this text is to demonstrate how to
38 how to split text into smaller chunks.
43 This is useful for indexing, searching, and
31 and processing large documents.
47 Each chunk will have a defined maximum size and
42 size and may overlap with adjacent chunks.


### PDF Document를 metadata가 보존된 chunk로 나누기

페이지별 `docs`를 실제 검색 단위인 chunk Document로 나눈다.

- 입력: 페이지 Document 6개가 담긴 `docs`이다.
- 설정: `chunk_size=200`, `chunk_overlap=35`이다.
- 출력: 원본 metadata가 복사된 chunk Document 목록이다.
- 다음 사용처: OpenAI Embedding과 Chroma의 입력이다.

이 셀은 결과를 다시 `docs`에 저장한다. 실행 후 `docs`의 의미는 페이지 목록에서 chunk 목록으로 바뀐다. 이 셀을 다시 실행하려면 먼저 PDF를 읽는 셀부터 실행해 페이지 목록을 복원한다.


In [20]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=35,
    separators=["\n\n", "\n", " ", ""]
)

docs = splitter.split_documents(docs)

print(len(docs)) # 13
for doc in docs:
    print(doc.page_content[:50], '...')

13
백설공주
옛날 어느 왕국에 공주님이 태어났어요.
“어쩜 이렇게 어여쁠까? 살결이 눈처럼 하 ...
왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 ...
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
시간이 흘러 백설공주는 어여쁜  ...
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.
하지만 사냥꾼은 차마 그럴 수 없었어요. ...
숲속을 헤매던 백설공주는 외딴 오두막에 이르렀어요.
들여다보니 오두막은 비어 있었어요.
“ ...
일곱 번째 침대에 쓰러져 잠들었어요.
밤이 되자 오두막 주인인 일곱 난쟁이가 돌아왔어요.
 ...
사람에게는 문을 열어 주지 마세요.”
며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아 ...
왕비는 먹음직스럽게 생긴 사과를 골라 독을 발랐어요.
그리고 과일 장수로 변장했지요.
왕비 ...
“난쟁이들이 문을 열어 주지 말라고 했어요.”
백설공주가 거절하자, 왕비는 창문 틈새로 사 ...
쓰러졌어요.
“호호호. 이제 내가 세상에서 가장 아름답겠지?”
왕비는 백설공주를 버려둔 채 ...
저녁이 되자, 일곱 난쟁이가 돌아왔어요.
난쟁이들은 쓰러진 백설공주를 보고 엉엉 울었어요. ...
어느 날, 한 왕자가 숲을 지나다가 유리관을 보았어요.
“누구지? 이 아름다운 여인은?”
 ...
왕자는 깨어난 백설공주를 보고 기뻐했어요.
“공주님, 나는 이웃 나라 왕자입니다.”
“왕자 ...


### 분할된 Document의 개수와 페이지 확인하기

분할 결과를 전수 출력해 검색 단위가 올바르게 만들어졌는지 확인한다.

- 전체 chunk 수가 원본 페이지 수보다 늘어났는지 확인한다.
- 각 chunk의 `metadata['page']`가 원래 페이지를 유지하는지 확인한다.
- `page_content`가 비어 있거나 지나치게 짧지 않은지 확인한다.

이 출력은 분할 결과의 페이지 보존을 확인한다. 출처 문자열은 뒤의 인용 함수에서 별도로 사용한다.


In [22]:
print(len(docs)) # 13
for index, doc in enumerate(docs):
    print(f'{index}: page-{doc.metadata["page"]}')
    print(doc.page_content[:20], '...')

13
0: page-0
백설공주
옛날 어느 왕국에 공주님이  ...
1: page-1
왕은 아름다운 새 왕비를 맞았어요.
 ...
2: page-1
그 대답을 들어야만 차가운 왕비 얼굴 ...
3: page-1
왕비는 사냥꾼에게 백설공주를 죽이라고 ...
4: page-2
숲속을 헤매던 백설공주는 외딴 오두막 ...
5: page-2
일곱 번째 침대에 쓰러져 잠들었어요. ...
6: page-2
사람에게는 문을 열어 주지 마세요.” ...
7: page-3
왕비는 먹음직스럽게 생긴 사과를 골라 ...
8: page-3
“난쟁이들이 문을 열어 주지 말라고  ...
9: page-3
쓰러졌어요.
“호호호. 이제 내가 세 ...
10: page-4
저녁이 되자, 일곱 난쟁이가 돌아왔어 ...
11: page-4
어느 날, 한 왕자가 숲을 지나다가  ...
12: page-5
왕자는 깨어난 백설공주를 보고 기뻐했 ...


### OpenAI 임베딩 객체 준비하기

Embedding은 텍스트의 의미를 숫자 배열로 바꾸는 과정이다. `OpenAIEmbeddings`는 이 변환을 수행하는 객체이다.

- 입력: Document 본문 또는 검색 질문 문자열이다.
- 출력: 의미를 나타내는 embedding vector이다.
- 모델: `text-embedding-3-small`을 사용한다.
- 다음 사용처: Chroma가 문서 vector와 질문 vector를 비교한다.

객체 생성만으로는 API가 호출되지 않는다. 실제 네트워크 요청과 비용은 다음 `Chroma.from_documents()`가 문서를 임베딩할 때 발생한다.


In [23]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

### chunk를 임베딩해 Chroma Vector Store 만들기

Vector Store는 Embedding Model이 아니다. 만들어진 vector와 원문 Document를 연결해 저장하고 검색하는 구성 요소이다.

`Chroma.from_documents()`는 다음 순서로 실행된다.

1. 각 Document의 `page_content`를 embedding vector로 변환한다.
2. vector와 원문·metadata를 함께 등록한다.
3. 검색 가능한 Chroma 객체를 반환한다.

- `documents`: 앞에서 만든 chunk Document 목록이다.
- `embedding`: 문서와 질문을 같은 의미 공간으로 바꾸는 객체이다.
- 반환값 `vector_store`: 다음 직접 검색과 Retriever 생성에 사용한다.

이 셀에서 OpenAI Embedding API 호출과 비용이 발생한다.


In [24]:
from langchain_chroma import Chroma

# 각 Document의 원문, vector, metadata를 함께 등록
vector_store = Chroma.from_documents(
    # 13개의 Chunk로 나누어진
    # Document(page_content, metadata) 목록
    documents=docs,

    # OpenAI Embedding API를 이용해서 벡터화를 수행하는 객체
    embedding=embeddings
)

# 반환된 vector_store는 메모리에 저장되어 있음
print(type(vector_store))


<class 'langchain_chroma.vectorstores.Chroma'>


### Vector Store를 직접 검색하고 거리 점수 읽기

`similarity_search_with_score()`는 질문과 가까운 chunk를 거리 점수와 함께 반환한다.

- 입력: 질문 문자열 `query`이다.
- 변환: 질문을 vector로 바꾸고 저장된 문서 vector와 비교한다.
- 출력: `[(Document, distance_score), ...]` 형태의 목록이다.
- 기본 개수: `k`를 생략하면 상위 4개를 반환한다.

현재 Chroma 결과에서는 거리 score가 작을수록 질문과 가깝다. 저장소나 거리 함수가 바뀌면 점수 방향도 다시 확인해야 한다.

점수만 보지 않고 상위 Document에 질문의 실제 근거 문장이 포함되는지 함께 읽는다.


In [25]:
query = '왕비와 백설공주 중에 누가 더 아름다울까?'

retrievals = vector_store.similarity_search_with_score(
    query=query,
    k=4 # 13개의 chunk 중 유사도가 가장 높은 4개만 조회(기본값 4)
)

for rank, (doc, score) in enumerate(retrievals, start=1):
    print(f'{rank}: {score}')
    print(doc.page_content[:100], '...')

1: 0.8956902027130127
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.
어느 날, 왕비는 요술 거울에게 물었지요.
“거울아, 거울아. 이 세상에 ...
2: 0.9641204476356506
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.
하지만 사냥꾼은 차마 그럴 수 없었어요.
“가여운 공주님, 왕비님이 찾지 못하도록 멀리멀리 떠
나세요.”
백설공주는 울면서 숲으로 ...
3: 0.966176450252533
사람에게는 문을 열어 주지 마세요.”
며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운
지 물었어요.
“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”
“사냥꾼 ...
4: 0.9730614423751831
왕은 아름다운 새 왕비를 맞았어요.
그런데 새 왕비는 자기보다 아름다운 사람을 두고 보
지 못했어요.
왕비는 진실만을 말하는 요술 거울에게 늘 이렇게 물
었어요.
“거울아, 거울아 ...


### Vector Store를 Retriever 인터페이스로 바꾸기

`Retriever`는 질문을 받아 관련 `list[Document]`를 반환하는 검색 랭체인 인터페이스이다. 문서를 저장하지 않고 LLM 답변도 만들지 않는다.

`as_retriever()`는 Chroma 검색 기능을 LangChain의 공통 호출 방식으로 감s싼다.

- `search_type='similarity'`: 의미적으로 가까운 문서를 선택한다.z
- `search_kwargs={'k': 3}`: 최대 3개의 Document를 반환한다.
- 기본 출력: 거리 score가 없는 `list[Document]`이다.

아래 코드는 `batch([query])`로 질문 목록을 전달한다. 반환값은 질문별 결과가 들어 있는 `list[list[Document]]`이며 첫 질문의 결과는 `[0]`으로 꺼낸다.


In [26]:

# Chroma를 Retriever로 감싸고 유사도 점수가 높은 3개만 조회
retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

# 입력(batch) : list[str]
# 출력(batch) : list[list[Document]]
retrievals = retriever.batch([query])

for rank, doc in enumerate(retrievals[0], start=1):
    print(f'{rank}')
    print(doc.page_content[:100], '...')

1
그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌
지요.
시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.
어느 날, 왕비는 요술 거울에게 물었지요.
“거울아, 거울아. 이 세상에 ...
2
왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.
하지만 사냥꾼은 차마 그럴 수 없었어요.
“가여운 공주님, 왕비님이 찾지 못하도록 멀리멀리 떠
나세요.”
백설공주는 울면서 숲으로 ...
3
사람에게는 문을 열어 주지 마세요.”
며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운
지 물었어요.
“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”
“사냥꾼 ...


## 2. 검색과 생성: 질문을 근거 문서와 함께 LLM에 전달하기

이제 질문이 들어온 뒤 실행되는 검색과 생성 단계를 만든다.

`질문 → Retriever → list[Document] → context → ChatPromptValue → AIMessage → 답변 문자열`

### 값의 역할

- `list[Document]`: Retriever가 고른 검색 결과이다.
- `context`: Document의 본문과 출처를 합친 프롬프트용 문자열이다.
- `ChatPromptValue`: 완성된 system·human message 묶음이다.
- `AIMessage`: Chat Model이 반환한 응답 객체이다.
- 답변 문자열: `AIMessage`에서 텍스트만 꺼낸 최종 생성 결과이다.

인용은 Python이 Document metadata에서 만든다. 검색·context 변환·생성·인용을 분리하면 잘못된 답변이 나온 단계를 추적하기 쉽다.


### 검색 근거를 받는 ChatPromptTemplate 만들기

`ChatPromptTemplate.from_messages()`는 역할이 다른 message 틀을 하나의 프롬프트로 만든다.

- `system`: 모델의 역할과 근거 제한 규칙을 모든 요청에 적용한다.
- `human`: 실행할 때 전달되는 질문과 검색 근거를 배치한다.
- `{query}`: 사용자의 질문 문자열이 들어갈 위치이다.
- `{context}`: 검색 Document를 합친 근거 문자열이 들어갈 위치이다.

context에서 답을 찾을 수 없으면 모른다고 말하도록 지시한다. 출처와 페이지는 뒤의 Python 함수가 metadata에서 만들기 때문에 모델이 추측하지 않는다.


In [27]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = (
    '당신은 어린 아이에게 꿈과 희망을 심어주는 유치원 교사이다. '
    '질문하는 아이의 반응에 최대한 호응하며 제공된 context만 근거로 답한다.'
)

HUMAN_PROMPT = '''제공된 context만 참고하여 질문에 답한다.
context에서 확인할 수 없는 내용은 모른다고 답한다.
답변에 필요한 사실만 간결하게 작성한다.

사용자 질문:
{query}

컨텍스트:
{context}

<<최종 응답 형식>>
답변: <<context에서 확인한 내용 또는 모른다는 설명>>'''


prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', HUMAN_PROMPT)
])


### Document metadata로 인용 표기 만들기

인용(citation)은 답변 근거의 출처와 위치를 표시하는 문자열이다. `document_citation()`은 Document 한 개를 받아 `snow-white.pdf (page: 4)` 형태의 문자열을 반환한다.

### 페이지 선택 순서

1. `page_label`이 있으면 사람이 보는 페이지 번호로 사용한다.
2. `page_label`이 없으면 0부터 시작하는 `page`에 1을 더한다.
3. 페이지 정보가 없으면 `unknown`을 사용한다.

이 규칙을 Python으로 고정하면 LLM이 파일명이나 페이지 번호를 추측하지 않는다.


In [28]:
from langchain_core.documents import Document

# citation: 인용
def document_citation(document: Document) -> str:

    # PDF 파일 경로
    source = document.metadata.get('source', 'unknown')

    # 사람이 보는 페이지 번호
    page_label = document.metadata.get('page_label')

    # page_label이 없을 경우
    if page_label is None:
        page_index = document.metadata.get('page')

        if isinstance(page_index,int): #  page가 있을 경우
            page_label = page_index + 1
        else:                          #  page가 없을 경우
            page_label = 'unknown'

    return f'{source} (page: {page_label})'

### Document 목록을 프롬프트용 context로 변환하기

`format_context()`는 검색 결과를 LLM이 읽을 수 있는 문자열로 바꾼다.

- 입력: Retriever가 반환한 `list[Document]`이다.
- 변환: 각 Document를 `[출처·페이지] + 본문` 문자열로 바꾼다.
- 결합: Document 사이에 구분선을 넣고 검색 순서대로 연결한다.
- 출력: Prompt의 `{context}`에 들어갈 문자열 하나이다.


In [31]:
# 검색 Document 목록을 출처·페이지·본문이 포함된 context 문자열 하나로 바꾼다.
def format_context(documents: list[Document]) -> str:

    # 하나의 List에 인용+본문이 str 요소로 저장 -> list[str]
    context_parts = [
        f'[{document_citation(document)}]\n{document.page_content}'
        for document in documents
    ]

    # join()은 chunk 사이에 구분선을 넣고 Prompt의 context에 전달할 str 하나를 반환한다.
    return '\n\n---\n\n'.join(context_parts) # list[str] -> str

### 실제 검색 결과로 모델 입력 미리보기

모델을 호출하기 전에 검색 결과와 완성된 message를 먼저 확인한다.

1. `retriever.invoke(query)`가 관련 `list[Document]`를 반환한다.
2. `format_context()`가 문서 목록을 context 문자열로 바꾼다.
3. `prompt.invoke()`가 context와 query를 system·human message에 배치한다.

`prompt.invoke()`는 LLM을 호출하지 않는다. 출력에서 출처·페이지·본문과 원래 질문이 올바른 message에 들어갔는지 확인한다. Retriever 검색에는 질문 임베딩을 위한 OpenAI API 호출이 발생한다.


In [32]:
# query (왕비와 백설공주 중 누가 더 예뻐?) 검색 결과(청크) 3개
preview_documents: list[Document] = retriever.invoke(query)

# 검색 결과 3개를 하나의 (인용 + 내용)이 반복되는 str로 변경
preview_context = format_context(preview_documents)

prompt_value = prompt.invoke({
    'query': query,
    'context': preview_context
})

prompt_value

ChatPromptValue(messages=[SystemMessage(content='당신은 어린 아이에게 꿈과 희망을 심어주는 유치원 교사이다. 질문하는 아이의 반응에 최대한 호응하며 제공된 context만 근거로 답한다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='제공된 context만 참고하여 질문에 답한다.\ncontext에서 확인할 수 없는 내용은 모른다고 답한다.\n답변에 필요한 사실만 간결하게 작성한다.\n\n사용자 질문:\n왕비와 백설공주 중에 누가 더 아름다울까?\n\n컨텍스트:\n[snow-white.pdf (page: 2)]\n그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌\n지요.\n시간이 흘러 백설공주는 어여쁜 소녀가 되었어요.\n어느 날, 왕비는 요술 거울에게 물었지요.\n“거울아, 거울아. 이 세상에서 누가 가장 아름답니?”\n“왕비님도 아름답지만 백설공주가 더 아름답습니다.”\n화가 난 왕비는 사냥꾼을 불렀어요.\n왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.\n\n---\n\n[snow-white.pdf (page: 2)]\n왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.\n하지만 사냥꾼은 차마 그럴 수 없었어요.\n“가여운 공주님, 왕비님이 찾지 못하도록 멀리멀리 떠\n나세요.”\n백설공주는 울면서 숲으로 도망쳤어요.\n\n---\n\n[snow-white.pdf (page: 3)]\n사람에게는 문을 열어 주지 마세요.”\n며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운\n지 물었어요.\n“왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.”\n“사냥꾼이 날 속였구나. 내가 직접 해치우겠어!”\n\n<<최종 응답 형식>>\n답변: <<context에서 확인한 내용 또는 모른다는 설명>>', additional_kwargs={}, response_metadata={})])

### 답변을 생성할 Chat Model 준비하기

`ChatOpenAI`는 역할별 message를 받아 `AIMessage`를 반환하는 LangChain Chat Model이다.

- `model=CHAT_MODEL_NAME`: 앞 환경 셀에서 정한 모델 ID를 사용한다.
- `use_responses_api=True`: OpenAI Responses API 경로로 요청한다.
- 반환값: 생성 텍스트와 부가 정보가 담긴 `AIMessage`이다.

객체 생성만으로는 API가 호출되지 않는다. 실제 네트워크 요청과 비용은 뒤에서 `run_rag()` 또는 `llm.invoke()`를 실행할 때 발생한다.


In [33]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=CHAT_MODEL_NAME,
    use_responses_api=True
)

### 검색과 생성을 2-step RAG 함수로 연결하기

2-step RAG는 검색을 먼저 수행하고 검색 결과를 한 번의 생성 단계에 전달한다. 아래 셀은 전체 흐름을 세 부분으로 구성한다.

### 생성 Chain

- `prompt | llm | StrOutputParser()` 순서로 연결한다.
- 입력은 `{'context': str, 'query': str}`이다.
- 출력은 답변 문자열 `str`이다.

### `run_rag()` 처리 순서

1. 질문으로 관련 `list[Document]`를 검색한다.
2. 문서 목록을 context 문자열로 바꾼다.
3. 생성 Chain으로 답변을 만든다.
4. `answer`와 `documents`를 하나의 dict로 반환한다.

### 결과 출력

`print_rag_result()`는 LLM 답변과 metadata 기반 참조문서를 나누어 출력한다. 답변은 모델이 만들지만 인용과 근거 본문은 Python이 실제 검색 Document에서 가져온다.


In [34]:
from langchain_core.output_parsers import StrOutputParser

# 1. chain 생성
generation_chain = prompt | llm | StrOutputParser()

# 2. 질문과 검색 결과를 생성 chain이 전달하는 함수
def run_rag(question:str) -> dict:

    # 2-1. question에 맞는 chunk document를 3개까지 찾기
    documents: list[Document] = retriever.invoke(question)

    # 2-2. Document 목록의 출처, 페이지 내용이 포함된
    # 하나의 context 문자열로 변환
    context = format_context(documents)

    # 2-3. query와 context를 generation_chain에 보내기
    answer: str = generation_chain.invoke({
        'context': context,
        'query': question
    })

    # 2-4. 생성 결과와 검색된 chunk document 반환
    return {
        'answer': answer,
        'documents': documents # 답변의 근거!!
    }

# 3. run_rag()가 반환한 답변과 근거를 화면에 출력
def print_rag_result(result: dict) -> None:
    # result는 {'answer': str, 'documents': list[Document]} 구조이다.
    print('답변:')
    print(result['answer'])
    print('\n참조문서:')

    for document in result['documents']:
        # 본문 앞 200자의 줄바꿈을 공백으로 바꾸어 답변과 근거를 한눈에 대조한다.
        evidence_preview = document.page_content[:200].replace('\n', ' ')
        print('-', document_citation(document))
        print('  근거:', evidence_preview)


# 4. 함수 호출
rag_result = run_rag(query)
print_rag_result(rag_result)

답변:
답변: 백설공주가 왕비보다 더 아름답다고 요술 거울이 말했어요.

참조문서:
- snow-white.pdf (page: 2)
  근거: 그 대답을 들어야만 차가운 왕비 얼굴에 미소가 번졌 지요. 시간이 흘러 백설공주는 어여쁜 소녀가 되었어요. 어느 날, 왕비는 요술 거울에게 물었지요. “거울아, 거울아. 이 세상에서 누가 가장 아름답니?” “왕비님도 아름답지만 백설공주가 더 아름답습니다.” 화가 난 왕비는 사냥꾼을 불렀어요. 왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요.
- snow-white.pdf (page: 2)
  근거: 왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요. 하지만 사냥꾼은 차마 그럴 수 없었어요. “가여운 공주님, 왕비님이 찾지 못하도록 멀리멀리 떠 나세요.” 백설공주는 울면서 숲으로 도망쳤어요.
- snow-white.pdf (page: 3)
  근거: 사람에게는 문을 열어 주지 마세요.” 며칠이 지나 왕비는 다시 요술 거울에게 누가 가장 아름다운 지 물었어요. “왕비님도 아름답지만 백설공주님이 천배는 더 아름답습니다.” “사냥꾼이 날 속였구나. 내가 직접 해치우겠어!”


### 문서에 명시된 사실 질문 확인하기

독사과의 색깔은 PDF에 직접 적혀 있는 사실이다.

- 답변이 `새빨간 색`을 언급하는지 확인한다.
- 참조문서에 `새빨간 사과`라는 근거가 있는지 확인한다.
- 답변과 인용 문장이 일치하는지 확인한다.

세 조건이 맞으면 검색과 생성이 올바르게 연결된 것이다.


In [35]:

apple_result = run_rag("백설공주가 먹은 독사과의 색깔은?")
print_rag_result(apple_result)

답변:
답변: 백설공주가 먹은 독사과는 새빨간 색깔이에요.

참조문서:
- snow-white.pdf (page: 3)
  근거: 숲속을 헤매던 백설공주는 외딴 오두막에 이르렀어요. 들여다보니 오두막은 비어 있었어요. “아무도 없네. 좀 쉬어 가도 될까? 어? 신기하다! 모든 게 작아.  어어? 이상하다! 모든 게 일곱. 의자도 일곱, 접시도 일곱. 어머,  침대도 일곱 개네.” 도망치느라 치진 백설공주는 식탁 위에 있던 빵을 먹고 나서 일곱 번째 침대에 쓰러져 잠들었어요.
- snow-white.pdf (page: 4)
  근거: “난쟁이들이 문을 열어 주지 말라고 했어요.” 백설공주가 거절하자, 왕비는 창문 틈새로 사과를 쑥 내밀었어 요. “그럼, 맛이라도 봐요. 정말 맛있으니까. 둘이 먹다 하나가 죽어 도 모를걸요.” “탐스러운 사과네. 맛있어 보여. 한입만 아삭 깨물어 볼까?” 사과를 베어 문 순간, 백설공주는 온몸에 독이 퍼져 정신을 잃고 쓰러졌어요.
- snow-white.pdf (page: 4)
  근거: 왕비는 먹음직스럽게 생긴 사과를 골라 독을 발랐어요. 그리고 과일 장수로 변장했지요. 왕비는 산을 넘고 또 넘어 일곱 난쟁이의 오두막에 도착했어요. “새콤달콤 맛있는 사과가 있어요. 아가씨의 붉은 입술처럼 새빨 간 사과랍니다. 잠깐 문을 열어 보세요.” 백설공주는 고개를 저었어요. “난쟁이들이 문을 열어 주지 말라고 했어요.”


### 여러 장면이 관련된 질문으로 RAG 답변 확인하기

왕자가 찾아왔을 때 백설공주가 있던 위치를 질문한다.

- 답변이 `유리관`을 언급하는지 확인한다.
- 해당 장면이 포함된 참조문서와 페이지가 표시되는지 확인한다.
- 다른 장면이 섞이면 Retriever가 고른 상위 Document부터 점검한다.


In [36]:
location_question = "왕자가 백설공주를 찾아왔을 때 백설공주는 어디에 있었나?"

rag_location_result = run_rag(location_question)
print_rag_result(rag_location_result)

답변:
답변: 왕자가 백설공주를 찾아왔을 때, 백설공주는 유리관 안에 있었어요.

참조문서:
- snow-white.pdf (page: 2)
  근거: 왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요. 하지만 사냥꾼은 차마 그럴 수 없었어요. “가여운 공주님, 왕비님이 찾지 못하도록 멀리멀리 떠 나세요.” 백설공주는 울면서 숲으로 도망쳤어요.
- snow-white.pdf (page: 6)
  근거: 왕자는 깨어난 백설공주를 보고 기뻐했어요. “공주님, 나는 이웃 나라 왕자입니다.” “왕자님이 나를 다시 살려 주셨군요.” “나와 결혼해 주시겠어요?” “네, 좋아요!” 두 사람은 일곱 난쟁이와 함께 오래오래 행복하게 살 았답니다.
- snow-white.pdf (page: 5)
  근거: 어느 날, 한 왕자가 숲을 지나다가 유리관을 보았어요. “누구지? 이 아름다운 여인은?” “백설공주랍니다.” 왕자는 백설공주에게 반해 유리관을 달라고 부탁했어요. 일곱 난쟁이는 백설공주를 잘 지킨다는 약속을 받고 유리관을 내주었지요. 그런데 신하들이 유리관을 옮기다 돌부리에 툭! 백설공주 목 에서 사과 조각이 툭!  “우아, 공주님이 살아났어!”


### 같은 질문을 검색 없이 LLM에 전달하기

같은 질문을 `llm.invoke()`에 직접 전달해 RAG 답변과 비교한다.

- RAG 경로: Retriever가 PDF context와 출처를 제공한다.
- 일반 LLM 경로: 모델이 학습 중 얻은 일반 지식만 사용한다.
- `.content`: 반환된 `AIMessage`에서 생성 텍스트를 꺼낸다.

두 답변이 자연스럽게 보여도 현재 PDF의 근거와 일치하는지는 별도로 확인해야 한다.


In [38]:
plain_message = llm.invoke(location_question)

print(plain_message.text)

백설공주는 독사과를 먹고 **유리관 속에 누워 잠들어 있었습니다.**


### 문서에 없는 내용은 모른다고 답하는지 확인하기

PDF에는 사냥꾼이 백설공주를 도운 장면만 있고 이후 행적은 없다.

- Retriever는 사냥꾼이 등장하는 관련 chunk를 찾을 수 있다.
- 그러나 검색된 문서에는 질문의 실제 답이 없다.
- 이 경우 모델은 후속 이야기를 만들지 않고 모른다고 답해야 한다.

관련 문서가 검색되었다는 사실과 질문의 답이 문서에 있다는 사실은 다르다. 최종 주장을 context에서 직접 확인해야 한다.


In [39]:
unknown_result = run_rag("백설공주를 살려준 사냥꾼은 그 후 어떻게 되었는가?")

print_rag_result(unknown_result)

답변:
답변: 사냥꾼은 백설공주를 멀리 떠나게 해 주었지만, 그 후 어떻게 되었는지는 알 수 없습니다.

참조문서:
- snow-white.pdf (page: 2)
  근거: 왕비는 사냥꾼에게 백설공주를 죽이라고 명령했어요. 하지만 사냥꾼은 차마 그럴 수 없었어요. “가여운 공주님, 왕비님이 찾지 못하도록 멀리멀리 떠 나세요.” 백설공주는 울면서 숲으로 도망쳤어요.
- snow-white.pdf (page: 1)
  근거: 백설공주 옛날 어느 왕국에 공주님이 태어났어요. “어쩜 이렇게 어여쁠까? 살결이 눈처럼 하얗구나. 백 설공주라고 불러야겠다.” 왕과 왕비는 갓 태어난 딸을 보며 기뻐했어요. 하지만 기쁨도 잠시, 왕비는 곧 세상을 떠나고 말았어 요.
- snow-white.pdf (page: 3)
  근거: 숲속을 헤매던 백설공주는 외딴 오두막에 이르렀어요. 들여다보니 오두막은 비어 있었어요. “아무도 없네. 좀 쉬어 가도 될까? 어? 신기하다! 모든 게 작아.  어어? 이상하다! 모든 게 일곱. 의자도 일곱, 접시도 일곱. 어머,  침대도 일곱 개네.” 도망치느라 치진 백설공주는 식탁 위에 있던 빵을 먹고 나서 일곱 번째 침대에 쓰러져 잠들었어요.
